# SageMaker Batch Inference Notebook

This notebook runs a simple proposal-aligned inference flow:
1. get latest model package
2. approve it (optional)
3. create small unseen inference input
4. run Batch Transform
5. curate predictions to Parquet in S3


In [ ]:
%pip install -q "sagemaker<3" pandas pyarrow s3fs


In [ ]:
import json
from datetime import datetime, timezone, timedelta
from io import BytesIO
from urllib.parse import urlparse

import boto3
import pandas as pd


In [ ]:
# Config
REGION = "us-east-1"
BUCKET = "agri-price-dev-raw"
ROLE_ARN = "arn:aws:iam::654654220088:role/LabRole"
MODEL_PACKAGE_GROUP = "agri-price-multi-output"

# Keep simple and fast for first run
LOOKBACK_DAYS = 30
AUTO_APPROVE_IF_NONE = True  # Set False if you always approve manually in console

run_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")

input_prefix = f"inference/input/run_date={run_date}/"
raw_output_prefix = f"predictions/raw/run_date={run_date}/"
curated_output_prefix = f"predictions/curated/run_date={run_date}/"

INPUT_S3 = f"s3://{BUCKET}/{input_prefix}"
RAW_OUTPUT_S3 = f"s3://{BUCKET}/{raw_output_prefix}"
CURATED_OUTPUT_S3 = f"s3://{BUCKET}/{curated_output_prefix}"

sm = boto3.client("sagemaker", region_name=REGION)
s3 = boto3.client("s3", region_name=REGION)

print("INPUT_S3:", INPUT_S3)
print("RAW_OUTPUT_S3:", RAW_OUTPUT_S3)
print("CURATED_OUTPUT_S3:", CURATED_OUTPUT_S3)


In [ ]:
# Get latest model package in group
resp = sm.list_model_packages(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP,
    SortBy="CreationTime",
    SortOrder="Descending",
    MaxResults=5,
)
pkgs = resp.get("ModelPackageSummaryList", [])
if not pkgs:
    raise RuntimeError(f"No model packages found in group {MODEL_PACKAGE_GROUP}")

print("Latest packages:")
for p in pkgs:
    print(p["ModelPackageArn"], p["ModelApprovalStatus"])

approved = [p for p in pkgs if p["ModelApprovalStatus"] == "Approved"]
if approved:
    model_package_arn = approved[0]["ModelPackageArn"]
else:
    latest_arn = pkgs[0]["ModelPackageArn"]
    if AUTO_APPROVE_IF_NONE:
        sm.update_model_package(ModelPackageArn=latest_arn, ModelApprovalStatus="Approved")
        model_package_arn = latest_arn
        print("Auto-approved:", model_package_arn)
    else:
        raise RuntimeError("No Approved model package found. Approve one, then rerun.")

print("Using model package:", model_package_arn)


In [ ]:
# Build small unseen inference input (last LOOKBACK_DAYS days)
features_path = f"s3://{BUCKET}/processed/features/"
df = pd.read_parquet(features_path)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)

cutoff = df["date"].max() - timedelta(days=LOOKBACK_DAYS)
df_small = df[df["date"] > cutoff].copy().reset_index(drop=True)
if df_small.empty:
    raise RuntimeError("No rows in lookback window. Increase LOOKBACK_DAYS.")

target_cols = [c for c in df_small.columns if c.startswith("target_next_day_")]
df_infer = df_small.drop(columns=target_cols, errors="ignore").copy()

# Keep manifest for curation alignment
input_manifest = pd.DataFrame({
    "row_id": range(len(df_small)),
    "date": pd.to_datetime(df_small["date"]).dt.strftime("%Y-%m-%d")
})

# JSONL lines: each line must match inference.py input_fn expectations
for c in df_infer.columns:
    if pd.api.types.is_datetime64_any_dtype(df_infer[c]):
        df_infer[c] = pd.to_datetime(df_infer[c]).dt.strftime("%Y-%m-%d")

records = df_infer.to_dict(orient="records")
jsonl = "\n".join(json.dumps({"instances": [r]}) for r in records) + "\n"

s3.put_object(Bucket=BUCKET, Key=f"{input_prefix}features.jsonl", Body=jsonl.encode("utf-8"))

manifest_buf = BytesIO()
input_manifest.to_parquet(manifest_buf, index=False)
s3.put_object(Bucket=BUCKET, Key=f"{input_prefix}input_manifest.parquet", Body=manifest_buf.getvalue())

print("input rows:", len(records))
print("input file:", f"s3://{BUCKET}/{input_prefix}features.jsonl")


In [ ]:
# Create model + start batch transform
model_name = f"agri-price-batch-model-{run_stamp}"
transform_job_name = f"agri-price-batch-predict-{run_stamp}"

sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=ROLE_ARN,
    Containers=[{"ModelPackageName": model_package_arn}],
)

sm.create_transform_job(
    TransformJobName=transform_job_name,
    ModelName=model_name,
    TransformInput={
        "DataSource": {
            "S3DataSource": {
                "S3DataType": "S3Prefix",
                "S3Uri": INPUT_S3,
            }
        },
        "ContentType": "application/json",
        "SplitType": "Line",
    },
    TransformOutput={
        "S3OutputPath": RAW_OUTPUT_S3,
        "Accept": "application/json",
        "AssembleWith": "Line",
    },
    TransformResources={
        "InstanceType": "ml.m5.large",
        "InstanceCount": 1,
    },
)

print("model_name:", model_name)
print("transform_job_name:", transform_job_name)


In [ ]:
# Wait for transform completion
sm.get_waiter("transform_job_completed_or_stopped").wait(TransformJobName=transform_job_name)
job = sm.describe_transform_job(TransformJobName=transform_job_name)
print("status:", job["TransformJobStatus"])
print("failure_reason:", job.get("FailureReason"))
print("output:", job["TransformOutput"]["S3OutputPath"])
if job["TransformJobStatus"] != "Completed":
    raise RuntimeError("Transform job did not complete successfully.")


In [ ]:
# Inspect raw output files
u = urlparse(RAW_OUTPUT_S3)
resp = s3.list_objects_v2(Bucket=u.netloc, Prefix=u.path.lstrip("/"))
raw_keys = [x["Key"] for x in resp.get("Contents", []) if x["Key"].endswith(".out")]
print("raw output files:", len(raw_keys))
for k in raw_keys[:10]:
    print(k)

if not raw_keys:
    all_keys = [x["Key"] for x in resp.get("Contents", [])]
    print("No .out files found. Keys under prefix:", all_keys[:20])
    raise RuntimeError("No transform output .out files found.")


In [ ]:
# Curate raw predictions to Parquet for Athena
pred_rows = []
for k in raw_keys:
    text = s3.get_object(Bucket=BUCKET, Key=k)["Body"].read().decode("utf-8")
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        payload = json.loads(line)
        for p in payload.get("predictions", []):
            pred_rows.append(p)

pred_df = pd.DataFrame(pred_rows).reset_index(drop=True)
if len(pred_df) != len(input_manifest):
    raise RuntimeError(f"Prediction rows ({len(pred_df)}) != input rows ({len(input_manifest)})")

curated = pd.DataFrame({
    "date": input_manifest["date"],
    "target_next_day_price_coriander_pred": pred_df["target_next_day_price_coriander"],
    "target_next_day_price_kale_pred": pred_df["target_next_day_price_kale"],
    "target_next_day_price_lime_pred": pred_df["target_next_day_price_lime"],
    "target_next_day_price_orange_pred": pred_df["target_next_day_price_orange"],
    "target_next_day_price_red_chili_pred": pred_df["target_next_day_price_red_chili"],
    "model_package_arn": model_package_arn,
    "run_date": run_date,
})

buf = BytesIO()
curated.to_parquet(buf, index=False)
s3.put_object(Bucket=BUCKET, Key=f"{curated_output_prefix}predictions.parquet", Body=buf.getvalue())

print(f"Wrote curated file: s3://{BUCKET}/{curated_output_prefix}predictions.parquet")
curated.head()


In [ ]:
print("Done. Next: Athena table/view for actual-vs-predicted using curated prefix.")
